# Transform Sprints Data
1. Read bronze `sprints` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `sprints` table

In [0]:
%run "/Workspace/Users/deepanshu.patil69@gmail.com/Formula1/common/01_Environmnet_config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
results_df = spark.read.table(bronze_table)

results_process_df = (
        results_df
            .drop("url")
            .withColumnsRenamed({
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName":"race_name",
                "positionText": "finish_position_text",
                "date": "race_date", 
                "grid": "grid_position", 
                "laps": "completed_laps", 
                "number": "car_number", 
                "position": "finish_position"
            })
)

results_process_df.display()

In [0]:
display(results_process_df.select(F.count('*')))

In [0]:
results_nxt_process_df = (
    results_process_df
        .dropna(subset=['season', 'round','constructor_id','driver_id'])
        .dropDuplicates(['season', 'round','constructor_id','driver_id'])
)

results_nxt_process_df.select(F.count('*')).display()

In [0]:
result_final_df = (
    results_nxt_process_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

result_final_df.display()

In [0]:
(
    result_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
%sql

select * from formula1.silver.sprints;